# LSK-CAViT — HAM10000 Training
**Enable GPU first: Settings → Accelerator → GPU P100**

Dataset needed: `skin-cancer-mnist-ham10000` (add via Add Input)

## Step 1: Install libraries

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','timm','einops','--quiet'])
print('Done.')

## Step 2: Check GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
else:
    print('NO GPU — go to Settings > Accelerator > GPU P100')

## Step 3: Check dataset

In [ ]:
from pathlib import Path
D = Path('/kaggle/input/skin-cancer-mnist-ham10000')
print('Dataset found:', D.exists())
if D.exists():
    for f in sorted(D.iterdir()): print(' ', f.name)

# Validate all required paths exist before training
IMG_DIR_1 = D / "ham10000_images_part1"
IMG_DIR_2 = D / "ham10000_images_part2"
META_CSV  = D / "HAM10000_metadata.csv"
print(f"\nIMG_DIR_1 exists: {IMG_DIR_1.exists()} → {IMG_DIR_1}")
print(f"IMG_DIR_2 exists: {IMG_DIR_2.exists()} → {IMG_DIR_2}")
print(f"META_CSV  exists: {META_CSV.exists()}  → {META_CSV}")
if not META_CSV.exists() or not IMG_DIR_1.exists():
    raise FileNotFoundError("Required dataset files missing — check folder names above and update IMG_DIR_1/IMG_DIR_2/META_CSV in Step 4 accordingly.")


## Step 4: Full training (8-15+ hours for 100 epochs / 5 folds)
This now exceeds Kaggle's ~9-hour interactive session limit and may also exceed the
12-hour "Save & Run All" background limit on some accelerators.

Recommendations:
- Use **Save Version → Save & Run All** for background execution (still required).
- If a run times out, the script saves `history_fold{N}.csv` after every epoch and
  a recovery checkpoint every 10 epochs (`checkpoint_fold{N}_ep{E}.pt`), so you can
  resume/restart that fold.
- If you hit a CUDA out-of-memory error around the pruning epoch (epoch 80) with
  `batch_size=32`, reduce it back to 16 in CFG — this changes effective batch size
  but not the epoch/pruning schedule, and should be noted if used.
- Consider running one fold per notebook session if a single session can't
  complete all 5 folds at 100 epochs each.

In [ ]:
# ── STEP 1: Imports ───────────────────────────────────────────────────────────
import os, json, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, accuracy_score, roc_auc_score,
                              confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

# ── STEP 2: Reproducibility seed ─────────────────────────────────────────────
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── STEP 3: Paths (Kaggle paths — adjust if running elsewhere) ────────────────
# On Kaggle the HAM10000 dataset is at:
DATA_DIR = Path("/kaggle/input/skin-cancer-mnist-ham10000")
IMG_DIR_1 = DATA_DIR / "ham10000_images_part1"
IMG_DIR_2 = DATA_DIR / "ham10000_images_part2"
META_CSV  = DATA_DIR / "HAM10000_metadata.csv"
OUTPUT_DIR = Path("/kaggle/working/lsk_cavit_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── STEP 4: Config ────────────────────────────────────────────────────────────
CFG = {
    # Model
    "img_size":    256,
    "patch_size":  16,
    "embed_dim":   768,
    "num_heads":   12,
    "num_layers":  12,
    "num_classes": 7,
    # Training
    "epochs":      100,         # matches paper's stated 100-epoch schedule
    "batch_size":  32,          # matches paper; NOTE: may risk CUDA OOM on P100
                                 #   at the pruning epoch — drop back to 16 if needed
    "lr":          3.2e-5,
    "weight_decay":0.023,
    "warmup_epochs":10,
    # Copula
    "alpha":       0.18,        # copula loss weight
    "lambda_cop":  0.12,        # attention bias scale
    # Pruning
    "prune_epoch": 80,          # matches paper's stated pruning epoch
    "prune_ratio": 0.55,
    # CV
    "n_folds":     5,
    "seed":        42,
}

# ── Which fold to run (0-indexed: 0=Fold1, 1=Fold2, ... 4=Fold5) ────────────
# Change this value and re-run the notebook for each fold.
FOLD_TO_RUN = 0   # 0 → Fold 1

# ── STEP 5: Class labels ──────────────────────────────────────────────────────
LABEL_MAP = {
    "nv":    0,   # Melanocytic Nevi
    "mel":   1,   # Melanoma
    "bkl":   2,   # Benign Keratosis
    "bcc":   3,   # Basal Cell Carcinoma
    "akiec": 4,   # Actinic Keratoses
    "vasc":  5,   # Vascular Lesions
    "df":    6,   # Dermatofibroma
}
CLASS_NAMES = ["nv", "mel", "bkl", "bcc", "akiec", "vasc", "df"]
NUM_CLASSES = 7

# ── STEP 6: Load metadata ─────────────────────────────────────────────────────
def load_metadata():
    df = pd.read_csv(META_CSV)
    df["label"] = df["dx"].map(LABEL_MAP)

    # Build image path for each row
    def get_img_path(img_id):
        p1 = IMG_DIR_1 / f"{img_id}.jpg"
        p2 = IMG_DIR_2 / f"{img_id}.jpg"
        if p1.exists(): return str(p1)
        if p2.exists(): return str(p2)
        return None

    df["img_path"] = df["image_id"].apply(get_img_path)
    df = df[df["img_path"].notna()].reset_index(drop=True)

    # Encode metadata features
    df["age_norm"]  = df["age"].fillna(df["age"].median()) / 90.0
    df["sex_enc"]   = (df["sex"] == "male").astype(float)
    site_dummies    = pd.get_dummies(df["localization"].fillna("unknown"),
                                     prefix="site")
    df = pd.concat([df, site_dummies], axis=1)
    site_cols = [c for c in df.columns if c.startswith("site_")]

    print(f"Loaded {len(df)} images")
    print(df["dx"].value_counts())
    return df, site_cols

df, SITE_COLS = load_metadata()
META_DIM = 2 + len(SITE_COLS)   # age + sex + site one-hots

# ── STEP 7: Dataset class ─────────────────────────────────────────────────────
class HAM10000Dataset(Dataset):
    def __init__(self, df, site_cols, transform=None):
        self.df        = df.reset_index(drop=True)
        self.site_cols = site_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(row["img_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)

        # Metadata vector: [age_norm, sex_enc, site_one_hots...]
        meta_vals = [row["age_norm"], row["sex_enc"]] + \
                    [row[c] for c in self.site_cols]
        meta = torch.tensor(meta_vals, dtype=torch.float32)

        label = torch.tensor(row["label"], dtype=torch.long)
        return img, meta, label

# ── STEP 8: Transforms ────────────────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(256),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── STEP 9: Gaussian Copula module ───────────────────────────────────────────
class GaussianCopula(nn.Module):
    """
    Estimates a 7x7 inter-class correlation matrix Sigma from
    soft predictions, then returns it as an attention bias.
    This is the core LCViT contribution from the paper.
    """
    def __init__(self, num_classes=7):
        super().__init__()
        self.num_classes = num_classes
        # Learnable correlation matrix (initialised to identity)
        self.register_buffer("sigma",
            torch.eye(num_classes))

    def estimate_sigma(self, probs):
        """
        probs: (N, C) soft predictions from training set
        Updates self.sigma via maximum pseudo-likelihood approximation.
        """
        # Convert to pseudo-observations via empirical CDF
        N, C = probs.shape
        ranks = torch.zeros_like(probs)
        for c in range(C):
            order = torch.argsort(probs[:, c])
            ranks[order, c] = torch.arange(1, N+1,
                                           dtype=torch.float32,
                                           device=probs.device) / (N + 1)
        # Clamp to avoid inf in normal quantile
        ranks = ranks.clamp(1e-6, 1 - 1e-6)

        # Normal quantile transform
        from torch.distributions import Normal
        normal = Normal(0, 1)
        z = normal.icdf(ranks)            # (N, C)

        # Pearson correlation of z → Sigma estimate
        z_c  = z - z.mean(0, keepdim=True)
        cov  = (z_c.T @ z_c) / (N - 1)
        std  = torch.sqrt(torch.diag(cov)).clamp(min=1e-8)
        sigma = cov / (std.unsqueeze(1) * std.unsqueeze(0))
        sigma = sigma.clamp(-0.999, 0.999)
        self.sigma = sigma.detach()

    def get_attention_bias(self, lambda_cop=0.12):
        """Returns (C, C) bias matrix to add to attention logits."""
        return lambda_cop * self.sigma

    def copula_consistency_loss(self, logits, targets):
        """
        KL-based copula consistency loss L_cop.
        Penalises divergence between predicted and one-hot copula distributions.
        """
        probs   = F.softmax(logits, dim=-1)           # (N, C)
        onehot  = F.one_hot(targets,
                             self.num_classes).float() # (N, C)
        # KL(pred || target) — averaged over batch
        log_p = torch.log(probs.clamp(min=1e-8))
        log_q = torch.log(onehot.clamp(min=1e-8))
        kl = (probs * (log_p - log_q)).sum(dim=-1).mean()
        return kl.clamp(max=10.0)   # safety clamp


# ── STEP 10: LSK-CAViT model ──────────────────────────────────────────────────
class LSKCAViT(nn.Module):
    """
    LSK-CAViT built on a Swin-V2-Base backbone (publicly available,
    matches the paper's parameter range) with:
      - Metadata cross-attention fusion (HAViT)
      - Copula-guided attention bias (LCViT)
    """
    def __init__(self, num_classes=7, meta_dim=20, embed_dim=768,
                 lambda_cop=0.12):
        super().__init__()
        self.lambda_cop = lambda_cop

        # ── Visual backbone: Swin-V2-Base pretrained on ImageNet-22k ──
        self.backbone = timm.create_model(
            "swinv2_base_window8_256",
            pretrained=True,
            num_classes=0,          # Remove classifier head
            img_size=256,           # must match model name (_256)
        )
        feat_dim = self.backbone.num_features   # 1024 for swinv2_base

        # ── Metadata encoder (2-layer MLP) ────────────────────────────
        self.meta_encoder = nn.Sequential(
            nn.Linear(meta_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, feat_dim),
            nn.LayerNorm(feat_dim),
        )

        # ── Cross-attention fusion (HAViT) ────────────────────────────
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=feat_dim,
            num_heads=8,
            batch_first=True,
            dropout=0.1,
        )
        self.cross_norm = nn.LayerNorm(feat_dim)

        # ── Copula module ─────────────────────────────────────────────
        self.copula = GaussianCopula(num_classes=num_classes)

        # ── Classifier head ───────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, images, meta):
        # 1. Visual features from backbone
        vis_feat = self.backbone(images)          # (B, feat_dim)

        # 2. Metadata features
        meta_feat = self.meta_encoder(meta)       # (B, feat_dim)

        # 3. Cross-attention fusion: visual queries, metadata keys/values
        vis_seq   = vis_feat.unsqueeze(1)         # (B, 1, feat_dim)
        meta_seq  = meta_feat.unsqueeze(1)        # (B, 1, feat_dim)
        fused, _  = self.cross_attn(vis_seq, meta_seq, meta_seq)
        fused     = self.cross_norm(fused.squeeze(1) + vis_feat)

        # 4. Copula bias applied to classifier logits
        logits = self.classifier(fused)           # (B, C)

        # Apply copula attention bias as a re-weighting of logits
        cop_bias = self.copula.get_attention_bias(self.lambda_cop)
        # cop_bias shape: (C, C) — weight each class by its copula affinity
        logits = logits + (logits @ cop_bias) * 0.1

        return logits


# ── STEP 11: Class-weighted sampler ──────────────────────────────────────────
def make_weighted_sampler(labels):
    class_counts = np.bincount(labels, minlength=NUM_CLASSES)
    class_weights = 1.0 / (class_counts + 1e-6)
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(
        weights=torch.from_numpy(sample_weights).float(),
        num_samples=len(sample_weights),
        replacement=True,
    )

# ── STEP 12: CutMix augmentation ─────────────────────────────────────────────
def cutmix(images, labels, alpha=1.0):
    if alpha <= 0 or random.random() > 0.5:
        return images, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    B   = images.size(0)
    idx = torch.randperm(B, device=images.device)

    W, H = images.size(3), images.size(2)
    cut_w = int(W * np.sqrt(1 - lam))
    cut_h = int(H * np.sqrt(1 - lam))
    cx = random.randint(0, W)
    cy = random.randint(0, H)
    x1 = max(cx - cut_w // 2, 0)
    x2 = min(cx + cut_w // 2, W)
    y1 = max(cy - cut_h // 2, 0)
    y2 = min(cy + cut_h // 2, H)

    images[:, :, y1:y2, x1:x2] = images[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return images, labels, labels[idx], lam

# ── STEP 13: Sparse pruning (SMP) ────────────────────────────────────────────
def apply_sparse_pruning(model, prune_ratio=0.55):
    """Magnitude-based global weight pruning — crash-safe version."""
    try:
        # Collect all weights onto CPU to avoid OOM during quantile computation
        all_weights = []
        for name, param in model.named_parameters():
            if "weight" in name and param.dim() >= 2:
                all_weights.append(param.data.cpu().abs().flatten())

        if not all_weights:
            print("  No weights found for pruning — skipping.")
            return model

        all_weights_cat = torch.cat(all_weights)
        threshold = torch.quantile(all_weights_cat,
                                   torch.tensor(float(prune_ratio))).item()

        pruned = 0
        total  = 0
        with torch.no_grad():
            for name, param in model.named_parameters():
                if "weight" in name and param.dim() >= 2:
                    mask = param.data.abs() >= threshold
                    param.data *= mask.float()
                    pruned += int((~mask).sum().item())
                    total  += mask.numel()

        print(f"  Pruned {pruned}/{total} weights "
              f"({100*pruned/total:.1f}%) at ratio={prune_ratio}")

        # Clear CUDA cache after pruning to free fragmented memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"  WARNING: Pruning failed ({e}) — continuing without pruning.")

    return model

# ── STEP 14: Training loop ────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, scheduler,
                    criterion, copula, alpha, device, epoch):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch_idx, (images, meta, labels) in enumerate(loader):
        images = images.to(device)
        meta   = meta.to(device)
        labels = labels.to(device)

        # CutMix
        images, labels_a, labels_b, lam = cutmix(images, labels)

        optimizer.zero_grad()
        logits = model(images, meta)

        # Combined loss: cross-entropy + copula consistency
        ce_loss = lam * criterion(logits, labels_a) + \
                  (1 - lam) * criterion(logits, labels_b)
        cop_loss = copula.copula_consistency_loss(logits, labels)
        loss = ce_loss + alpha * cop_loss

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

        if batch_idx % 50 == 0:
            print(f"  Batch {batch_idx}/{len(loader)} "
                  f"loss={loss.item():.4f}")

    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    avg_loss = total_loss / len(loader)
    return avg_loss, f1


@torch.no_grad()
def evaluate(model, loader, criterion, copula, alpha, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []

    for images, meta, labels in loader:
        images = images.to(device)
        meta   = meta.to(device)
        labels = labels.to(device)

        logits = model(images, meta)
        ce_loss = criterion(logits, labels)
        cop_loss = copula.copula_consistency_loss(logits, labels)
        loss = ce_loss + alpha * cop_loss

        total_loss += loss.item()
        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)

    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs,
                             multi_class="ovr", average="macro")
    except Exception:
        auc = 0.0

    avg_loss = total_loss / len(loader)
    return avg_loss, acc, f1, auc, all_labels, all_preds, all_probs


# ── STEP 15: Cross-validation sigma update ───────────────────────────────────
@torch.no_grad()
def update_sigma(model, loader, device):
    """Collect training-set predictions to re-estimate copula Sigma."""
    model.eval()
    all_probs = []
    for images, meta, _ in loader:
        images = images.to(device)
        meta   = meta.to(device)
        logits = model(images, meta)
        probs  = F.softmax(logits, dim=1)
        all_probs.append(probs)
    all_probs = torch.cat(all_probs, dim=0)
    model.copula.estimate_sigma(all_probs)
    model.train()


# ── STEP 16: Save confusion matrix ───────────────────────────────────────────
def save_confusion_matrix(labels, preds, fold, output_dir):
    cm = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"Normalised Confusion Matrix — Fold {fold+1}")
    plt.tight_layout()
    path = output_dir / f"confusion_matrix_fold{fold+1}.png"
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"  Saved: {path}")


# ── STEP 17: Main 5-fold training ─────────────────────────────────────────────
def main():
    print("\n" + "="*60)
    print("LSK-CAViT: Starting 5-Fold Cross-Validation")
    print("="*60)

    skf = StratifiedKFold(n_splits=CFG["n_folds"],
                          shuffle=True, random_state=CFG["seed"])
    labels_array = df["label"].values
    img_paths    = df["img_path"].values

    fold_results = []
    all_fold_labels = []
    all_fold_preds  = []

    # Save split indices for reproducibility (Supplementary File S1)
    split_indices = []

    for fold, (train_idx, val_idx) in enumerate(
            skf.split(img_paths, labels_array)):

        # ── Run only the selected fold ────────────────────────────────────
        if fold != FOLD_TO_RUN:
            continue

        split_indices.append({
            "fold": fold,
            "train": train_idx.tolist(),
            "val":   val_idx.tolist()
        })

        print(f"\n{'─'*50}")
        print(f"FOLD {fold+1} / {CFG['n_folds']}")
        print(f"  Train: {len(train_idx)} | Val: {len(val_idx)}")
        print(f"{'─'*50}")

        set_seed(CFG["seed"] + fold)

        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df   = df.iloc[val_idx].reset_index(drop=True)

        train_ds = HAM10000Dataset(train_df, SITE_COLS, train_transform)
        val_ds   = HAM10000Dataset(val_df,   SITE_COLS, val_transform)

        sampler = make_weighted_sampler(train_df["label"].values)
        train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                                  sampler=sampler, num_workers=0,  # set to 0 to reduce memory pressure
                                  pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"],
                                  shuffle=False, num_workers=0,  # set to 0 to reduce memory pressure
                                  pin_memory=True)

        # Model
        model = LSKCAViT(
            num_classes=NUM_CLASSES,
            meta_dim=META_DIM,
            embed_dim=CFG["embed_dim"],
            lambda_cop=CFG["lambda_cop"],
        ).to(DEVICE)

        copula = model.copula

        # Class weights for loss
        class_counts = np.bincount(train_df["label"].values,
                                    minlength=NUM_CLASSES)
        class_wts = torch.tensor(
            len(train_df) / (NUM_CLASSES * class_counts + 1e-6),
            dtype=torch.float32
        ).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_wts,
                                         label_smoothing=0.1)

        # Optimiser + scheduler
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=CFG["lr"],
            weight_decay=CFG["weight_decay"]
        )
        total_steps  = CFG["epochs"] * len(train_loader)
        warmup_steps = CFG["warmup_epochs"] * len(train_loader)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=CFG["lr"],
            total_steps=total_steps,
            pct_start=warmup_steps / total_steps,
        )

        best_f1    = 0.0
        best_state = None
        history    = []

        for epoch in range(CFG["epochs"]):
            print(f"\n  Epoch {epoch+1}/{CFG['epochs']}")

            # Re-estimate copula Sigma every epoch on training set
            if epoch > 0 and epoch % 5 == 0:
                print("  Updating copula Sigma...")
                update_sigma(model, train_loader, DEVICE)

            # Apply pruning at scheduled epoch
            if epoch == CFG["prune_epoch"]:
                print(f"  Applying SMP (ratio={CFG['prune_ratio']})...")
                try:
                    model = apply_sparse_pruning(model, CFG["prune_ratio"])
                    model = model.to(DEVICE)
                    # Rebuild copula reference after model reassignment
                    copula = model.copula
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    print("  SMP complete. GPU cache cleared.")
                except Exception as e:
                    print(f"  WARNING: Pruning step failed ({e}) — skipping pruning for this fold.")

            tr_loss, tr_f1 = train_one_epoch(
                model, train_loader, optimizer, scheduler,
                criterion, copula, CFG["alpha"], DEVICE, epoch
            )
            val_loss, val_acc, val_f1, val_auc, _, _, _ = evaluate(
                model, val_loader, criterion, copula,
                CFG["alpha"], DEVICE
            )

            history.append({
                "epoch": epoch+1,
                "tr_loss": tr_loss, "tr_f1": tr_f1,
                "val_loss": val_loss, "val_acc": val_acc,
                "val_f1": val_f1, "val_auc": val_auc,
            })

            # Save history CSV after every epoch (crash recovery)
            pd.DataFrame(history).to_csv(
                OUTPUT_DIR / f"history_fold{fold+1}.csv", index=False)

            # Save checkpoint every 10 epochs (crash recovery)
            if (epoch + 1) % 10 == 0 and best_state is not None:
                ckpt_path = OUTPUT_DIR / f"checkpoint_fold{fold+1}_ep{epoch+1}.pt"
                torch.save(best_state, ckpt_path)
                print(f"  Checkpoint saved: {ckpt_path.name}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            print(f"  Train  → loss={tr_loss:.4f}  F1={tr_f1:.4f}")
            print(f"  Val    → loss={val_loss:.4f}  "
                  f"Acc={val_acc:.4f}  F1={val_f1:.4f}  AUC={val_auc:.4f}")

            if val_f1 > best_f1:
                best_f1    = val_f1
                best_state = {k: v.cpu().clone()
                              for k, v in model.state_dict().items()}
                print(f"  ★ New best F1={best_f1:.4f} — saving checkpoint")

        # ── Final evaluation on best checkpoint ───────────────────────
        model.load_state_dict({k: v.to(DEVICE)
                                for k, v in best_state.items()})
        _, val_acc, val_f1, val_auc, fold_labels, fold_preds, _ = evaluate(
            model, val_loader, criterion, copula, CFG["alpha"], DEVICE
        )

        fold_results.append({
            "fold": fold+1, "acc": val_acc,
            "f1": val_f1, "auc": val_auc
        })
        all_fold_labels.extend(fold_labels)
        all_fold_preds.extend(fold_preds)

        print(f"\n  FOLD {fold+1} FINAL → "
              f"Acc={val_acc:.4f}  F1={val_f1:.4f}  AUC={val_auc:.4f}")

        # Save model + confusion matrix
        torch.save(best_state,
                   OUTPUT_DIR / f"lsk_cavit_fold{fold+1}.pt")
        save_confusion_matrix(fold_labels, fold_preds, fold, OUTPUT_DIR)

        # Save history
        pd.DataFrame(history).to_csv(
            OUTPUT_DIR / f"history_fold{fold+1}.csv", index=False)

    # ── Save split file (Supplementary S1) ───────────────────────────
    with open(OUTPUT_DIR / "cv_splits_seed42.json", "w") as f:
        json.dump(split_indices, f)
    print("\nSaved CV split indices → cv_splits_seed42.json")

    # ── Summary ───────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("5-FOLD CROSS-VALIDATION SUMMARY")
    print("="*60)
    results_df = pd.DataFrame(fold_results)
    print(results_df.to_string(index=False))
    print(f"\nMean Accuracy : {results_df['acc'].mean():.4f} "
          f"± {results_df['acc'].std():.4f}")
    print(f"Mean Macro-F1 : {results_df['f1'].mean():.4f} "
          f"± {results_df['f1'].std():.4f}")
    print(f"Mean AUC      : {results_df['auc'].mean():.4f} "
          f"± {results_df['auc'].std():.4f}")

    results_df.to_csv(OUTPUT_DIR / "fold_results.csv", index=False)

    # Overall classification report
    print("\nOverall Classification Report (all folds pooled):")
    print(classification_report(all_fold_labels, all_fold_preds,
                                  target_names=CLASS_NAMES,
                                  zero_division=0))

    # Overall confusion matrix
    save_confusion_matrix(np.array(all_fold_labels),
                          np.array(all_fold_preds),
                          fold="all", output_dir=OUTPUT_DIR)

    print(f"\nAll outputs saved to: {OUTPUT_DIR}")
    return results_df


if __name__ == "__main__":
    results = main()


## Step 5: View results

In [ ]:
import pandas as pd
from pathlib import Path
OUT = Path('/kaggle/working/lsk_cavit_outputs')
if (OUT/'fold_results.csv').exists():
    r = pd.read_csv(OUT/'fold_results.csv')
    print(r.to_string(index=False))
    print(f"\nMean Acc: {r['acc'].mean():.4f} +/- {r['acc'].std():.4f}")
    print(f"Mean F1:  {r['f1'].mean():.4f} +/- {r['f1'].std():.4f}")
    print(f"Mean AUC: {r['auc'].mean():.4f} +/- {r['auc'].std():.4f}")
else:
    print('Training not finished yet.')

In [ ]:
from IPython.display import Image as IPImg, display
from pathlib import Path
OUT = Path('/kaggle/working/lsk_cavit_outputs')
for fold in range(1,6):
    p = OUT/f'confusion_matrix_fold{fold}.png'
    if p.exists():
        print(f'Fold {fold}:')
        display(IPImg(str(p)))

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/lsk_cavit_results','zip','/kaggle/working/lsk_cavit_outputs')
print('Zipped. Download lsk_cavit_results.zip from the Output panel.')

## Step 6: Output files

All files are saved to `/kaggle/working/lsk_cavit_outputs/` and bundled into
`lsk_cavit_results.zip` (downloadable from the Kaggle output panel).

| File | Description |
|------|-------------|
| `lsk_cavit_fold1.pt` … `lsk_cavit_fold5.pt` | Trained model weights (best F1 checkpoint) for each fold |
| `checkpoint_fold{N}_ep{E}.pt` | Crash-recovery checkpoints saved every 10 epochs per fold |
| `history_fold1.csv` … `history_fold5.csv` | Per-epoch train/val loss, accuracy, F1 for each fold |
| `confusion_matrix_fold1.png` … `confusion_matrix_fold5.png` | Per-fold confusion matrix plots |
| `confusion_matrix_foldall.png` | Aggregated confusion matrix across all 5 folds |
| `fold_results.csv` | Summary: Accuracy, F1, AUC for each of the 5 folds + mean/std |
| `cv_splits_seed42.json` | Train/val index splits used (reproducibility) |
| `lsk_cavit_results.zip` | All of the above bundled for download |
